# Embeddings e tópicos de notícias

| Etapa | O que faz | Proteção |
|---|---|---|
| **0. Embeddings** | título → `paraphrase-multilingual-mpnet-base-v2` → `news_events.embedding` | pula artigos já embedados |
| **1. Explorar espaço** | UMAP dos embeddings, distribuição temporal | sempre |
| **2. BERTopic** | topic modeling no corpus completo | re-roda com novo `run_id` |
| **3. Tópicos** | distribuição, palavras-chave, evolução temporal | sempre |

> Modelo multilingual — funciona para os 73 países sem tradução prévia.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import text
from db import engine, admin_engine
from style import GREEN, MUTED, setup_style
setup_style()

MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'  # 768-dim, multilingual
BATCH_SIZE  = 256   # artigos por batch de embedding
RUN_ID          = pd.Timestamp.now().strftime('%Y-%m-%dT%H:%M')  # identificador do run BERTopic
STOP_WORDS_LANG = 'english'  # corpus atual é predominantemente em inglês
NEWS_LANG       = 'en'       # trocar para 'multilingual' quando a fonte de notícias mudar

print(f'Run ID  : {RUN_ID}')
print(f'Modelo  : {MODEL_NAME}')

total = pd.read_sql('SELECT count(*) AS n FROM news_events', engine).iloc[0]['n']
embedded = pd.read_sql('SELECT count(*) AS n FROM news_events WHERE embedding IS NOT NULL', engine).iloc[0]['n']
print(f'Artigos : {total:,} total  ·  {embedded:,} já embedados  ·  {total - embedded:,} pendentes')

## 0. Embeddings

Computa embeddings do título de cada artigo e salva em `news_events.embedding`.
Pula os já embedados — seguro re-executar.

In [ ]:
from sentence_transformers import SentenceTransformer

pendentes = pd.read_sql("""
    SELECT id, title FROM news_events
    WHERE  embedding IS NULL
    ORDER  BY published_at
""", engine)

if pendentes.empty:
    print('Todos os artigos já têm embedding.')
else:
    print(f'Carregando modelo {MODEL_NAME}...')
    model = SentenceTransformer(MODEL_NAME)
    print(f'Modelo carregado. Computando embeddings para {len(pendentes):,} artigos...')

    ids    = pendentes['id'].tolist()
    titles = pendentes['title'].tolist()

    for i in range(0, len(titles), BATCH_SIZE):
        batch_ids    = ids[i:i + BATCH_SIZE]
        batch_titles = titles[i:i + BATCH_SIZE]

        vecs = model.encode(batch_titles, show_progress_bar=False, normalize_embeddings=True)

        with admin_engine.begin() as conn:
            for article_id, vec in zip(batch_ids, vecs):
                conn.execute(
                    text('UPDATE news_events SET embedding = :v WHERE id = :id'),
                    {'v': str(vec.tolist()), 'id': int(article_id)}
                )

        done = min(i + BATCH_SIZE, len(titles))
        print(f'  {done:,}/{len(titles):,}  ({done/len(titles)*100:.0f}%)', end='\r')

    print(f'\nEmbeddings salvos para {len(pendentes):,} artigos.')

## 1. Espaço de embeddings

UMAP dos embeddings colorido por sentimento e por tópico (após seção 2).

In [ ]:
df_emb = pd.read_sql("""
    SELECT id, title, published_at::date AS data,
           overall_sentiment_label AS sentimento,
           embedding::text          AS emb_text
    FROM   news_events
    WHERE  embedding IS NOT NULL
    ORDER  BY published_at
""", engine)

if df_emb.empty:
    print('Sem embeddings — execute a seção 0 primeiro.')
else:
    import ast
    print(f'{len(df_emb):,} artigos com embedding')
    # parse embeddings
    X = np.array([ast.literal_eval(e) for e in df_emb['emb_text']], dtype=np.float32)
    print(f'Shape: {X.shape}')

In [ ]:
if not df_emb.empty:
    import umap
    print('Calculando UMAP 2D...')
    reducer  = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                         metric='cosine', random_state=42, low_memory=True)
    coords   = reducer.fit_transform(X)
    df_emb['umap_x'] = coords[:, 0]
    df_emb['umap_y'] = coords[:, 1]

    # colorir por sentimento
    cores = {
        'Bullish': GREEN, 'Somewhat-Bullish': '#57e38a',
        'Bearish': '#ef4444', 'Somewhat-Bearish': '#f97316',
        'Neutral': '#9ca3af'
    }
    fig, ax = plt.subplots(figsize=(11, 8))
    for sent, grp in df_emb.groupby('sentimento'):
        ax.scatter(grp.umap_x, grp.umap_y, s=8, alpha=0.5,
                   color=cores.get(sent, MUTED), label=sent, edgecolors='none')
    ax.set_title('UMAP dos embeddings de notícias — cor = sentimento')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(markerscale=3, fontsize=8)
    plt.tight_layout()
    plt.show()

## 2. BERTopic

Modela tópicos no corpus completo de notícias. Cada run gera um `run_id` único — re-rodar não sobrescreve runs anteriores.

In [ ]:
if not df_emb.empty:
    from bertopic import BERTopic
    from sklearn.feature_extraction.text import CountVectorizer

    print(f'Rodando BERTopic em {len(df_emb):,} artigos...')
    vectorizer = CountVectorizer(stop_words=STOP_WORDS_LANG, min_df=3, ngram_range=(1, 2))
    topic_model = BERTopic(
        vectorizer_model     = vectorizer,
        nr_topics            = 'auto',
        calculate_probabilities = False,
        verbose              = True,
    )
    topics, _ = topic_model.fit_transform(df_emb['title'].tolist(), X)
    df_emb['topic_id'] = topics

    info = topic_model.get_topic_info()
    print(f'\n{len(info) - 1} tópicos encontrados  (excl. outliers -1)')
    print(info[info.Topic != -1].head(10).to_string(index=False))

In [ ]:
import json
# salvar definições de tópicos e assignments no banco
if not df_emb.empty and "topic_id" in df_emb.columns:
    from sqlalchemy import text

    topic_info = topic_model.get_topic_info()
    run_config = {
        "embedding_model": MODEL_NAME,
        "news_lang":       NEWS_LANG,
        "stop_words":      STOP_WORDS_LANG,
        "corpus_size":     len(df_emb),
    }

    with admin_engine.begin() as conn:
        for _, row in topic_info.iterrows():
            tid        = int(row.Topic)
            words      = topic_model.get_topic(tid)
            words_json = str([{"word": w, "score": round(s, 4)} for w, s in (words or [])])
            conn.execute(text("""
                INSERT INTO news_topic_definitions (run_id, topic_id, label, top_words, article_count, config)
                VALUES (:run, :tid, :label, :words::jsonb, :cnt, :cfg::jsonb)
                ON CONFLICT (run_id, topic_id) DO UPDATE SET
                    label = EXCLUDED.label, top_words = EXCLUDED.top_words,
                    article_count = EXCLUDED.article_count, config = EXCLUDED.config
            """), {"run": RUN_ID, "tid": tid,
                   "label": str(row.get("Name", row.get("Representation", ""))),
                   "words": words_json, "cnt": int(row.Count),
                   "cfg": json.dumps(run_config)})

        for _, row in df_emb.iterrows():
            conn.execute(text("""
                UPDATE news_events SET topic_id = :tid, topic_run = :run WHERE id = :id
            """), {"tid": int(row.topic_id), "run": RUN_ID, "id": int(row.id)})

    print(f"Tópicos salvos (run_id={RUN_ID}  ·  {NEWS_LANG}  ·  stop_words={STOP_WORDS_LANG})")


## 3. Explorar tópicos

In [ ]:
topicos = pd.read_sql("""
    SELECT d.topic_id, d.label, d.article_count, d.top_words
    FROM   news_topic_definitions d
    WHERE  d.run_id = (SELECT max(run_id) FROM news_topic_definitions)
      AND  d.topic_id <> -1
    ORDER  BY d.article_count DESC
""", engine)

if topicos.empty:
    print('Sem tópicos — execute a seção 2 primeiro.')
else:
    print(f'{len(topicos)} tópicos')

    fig, ax = plt.subplots(figsize=(12, max(4, len(topicos) * 0.3)))
    ax.barh(topicos['label'].str[:50], topicos['article_count'], color=GREEN, edgecolor='none')
    ax.set_xlabel('Artigos')
    ax.set_title('Tópicos por volume de artigos')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# UMAP colorido por tópico
if not df_emb.empty and 'topic_id' in df_emb.columns:
    n_t     = df_emb['topic_id'].nunique()
    palette = plt.cm.get_cmap('tab20', n_t)
    tid_map = {t: i for i, t in enumerate(sorted(df_emb['topic_id'].unique()))}

    fig, ax = plt.subplots(figsize=(11, 8))
    for tid, grp in df_emb.groupby('topic_id'):
        label = f't{tid}' if tid != -1 else 'outlier'
        ax.scatter(grp.umap_x, grp.umap_y, s=8, alpha=0.5,
                   color=palette(tid_map[tid]), label=label, edgecolors='none')
    ax.set_title(f'UMAP dos embeddings — cor = tópico BERTopic')
    ax.set_xlabel('UMAP 1')
    ax.set_ylabel('UMAP 2')
    ax.legend(markerscale=3, fontsize=7, ncol=4, loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
# evolução dos tópicos ao longo do tempo
if not df_emb.empty and 'topic_id' in df_emb.columns:
    df_time = df_emb[df_emb['topic_id'] != -1].copy()
    df_time['semana'] = pd.to_datetime(df_time['data']).dt.to_period('W').dt.start_time
    pivot = (df_time.groupby(['semana', 'topic_id'])
             .size().unstack(fill_value=0))

    top_topics = df_time['topic_id'].value_counts().head(6).index
    fig, ax = plt.subplots(figsize=(13, 5))
    for tid in top_topics:
        if tid in pivot.columns:
            ax.plot(pivot.index, pivot[tid], linewidth=1.5, label=f't{tid}')
    ax.set_xlabel('Semana')
    ax.set_ylabel('Artigos')
    ax.set_title('Evolução dos top 6 tópicos ao longo do tempo')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()